# BirdCLEF+ 2025 - Pretrained models: adaptive batch normalisation (AdaBN) on our team's baseline v2

**T4 GPU, internet off.** This notebook re-estimates only the BatchNorm running statistics of our team's trained baseline v2 model (`efficientnet_b0`, ImageNet-pretrained, Li et al. 2018) on the unlabelled `train_soundscapes`. There are no labels and no gradients: every weight stays as trained, and only `running_mean`, `running_var` and `num_batches_tracked` change. We use the same procedure as in the A1 AdaBN notebook, so that we can compare AdaBN's effect on ImageNet- and bird-pretrained backbones.

The windows come from the live audio through the same `decode`, `windows_to_inputs` and `to_input` as scored inference (norm branch `v2`). Window generation stays in the 4-process CPU pool, and the model runs on the GPU in full float32 (no AMP). Validation runs twice, before and after adaptation, with the baseline's split and evaluation on the cached mels. We check the value before adaptation against the baseline's recorded best validation AUC.

Set `M3_SMOKE_TEST=1` for a local run on synthetic data.


In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core
import time
T_NOTEBOOK = time.time()

import torch
SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"          # local tests only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE:
    print("M3_SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")


In [ ]:
import timm
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("DEVICE:", DEVICE, "|", DEVICE_NAME)

def gpu_sync():
    # CUDA calls return before the work finishes; wait for it so that timings are real.
    if DEVICE == "cuda":
        torch.cuda.synchronize()

# ---- configuration ----
NUM_WORKERS = 4
VAL_WORKERS = int(os.environ.get("M3_VAL_WORKERS", "4"))
TORCH_THREADS = 4
SEED = 42
BATCH_SIZE = int(os.environ.get("M3_BATCH_SIZE", "64"))  # overridden only by the local test
GUARD_BATCHES = 8
GUARD_MIN = float(os.environ.get("M3_GUARD_MIN", "30"))  # stop if the projection exceeds this
MIN_WINDOWS = 11000
VAL_BEFORE_TOL = 0.002           # allowed gap between the unadapted val AUC here and M1's recorded value
SOURCE_KERNEL = "sc4000-birdclef-m1-train-v2"

import glob, importlib, json, multiprocessing as mp, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

WORK = Path(os.environ.get("M3_WORK_DIR", "."))
WORK.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    if os.environ.get("M3_DATA_DIR"):       # local tests only
        return Path(os.environ["M3_DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    return Path(hits[0]).parent if hits else Path("../../data")

def find_model_dir(kernel):
    if os.environ.get("M3_MODEL_DIR"):      # local tests only
        return Path(os.environ["M3_MODEL_DIR"])
    hits = [h for h in glob.glob("/kaggle/input/**/model.pth", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise RuntimeError(f"expected one model.pth under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

def find_mels_dir():
    if os.environ.get("M3_MELS_DIR"):       # local tests only
        return Path(os.environ["M3_MELS_DIR"])
    hits = glob.glob("/kaggle/input/**/train_mels.npy", recursive=True)
    if not hits:
        raise RuntimeError("train_mels.npy not found; attach jmemei/sc4000-birdclef-m1-make-mels")
    return Path(hits[0]).parent

DATA, MODEL_DIR, MELS = find_data_dir(), find_model_dir(SOURCE_KERNEL), find_mels_dir()
CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
CFG.setdefault("norm", "v2"); CFG.setdefault("norm_top_db", 80.0)   # M1 cfg.json has neither (same fallback as fast inference)
print("cfg norm:", CFG["norm"], "| norm_top_db:", CFG["norm_top_db"], "| model:", CFG["model"], "| best_val_auc in cfg:", CFG.get("best_val_auc"))
assert CFG["norm"] == "v2", f"expected norm v2, cfg.json has {CFG['norm']}"
assert float(CFG["norm_top_db"]) == 80.0, f"expected norm_top_db 80, cfg.json has {CFG['norm_top_db']}"
assert CFG["model"] == "efficientnet_b0", f"expected efficientnet_b0, cfg.json has {CFG['model']}"
sample_sub_cols = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
assert sample_sub_cols == species, "species order differs from sample_submission"
print("Data:", DATA, "| model:", MODEL_DIR, "| mels:", MELS, "| CPUs:", os.cpu_count(), "| smoke:", SMOKE)


## Worker code
We copied `to_input`, `decode` and `windows_to_inputs` verbatim from the fast-infer notebook. The module is written to disk so that `spawn` workers can import it.


In [ ]:
LIB_SRC = r"""
import numpy as np
import soundfile as sf
import librosa

CFG, WIN, TOP_DB = None, None, None

def configure(cfg):
    global CFG, WIN, TOP_DB
    CFG, WIN, TOP_DB = cfg, cfg["sr"] * cfg["duration"], float(cfg["norm_top_db"])

def to_input(db):
    # Same as to_input in the A1 notebooks. For norm "v2" this equals M1's (db + 80) / 80.
    db = np.maximum(db - db.max(), -TOP_DB)
    if CFG["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + TOP_DB) / TOP_DB).astype(np.float32)

def decode(path):
    y, _ = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    return y.reshape(n_win, WIN)

def windows_to_inputs(batch):
    # One melspectrogram call for the file. Each row gets its own STFT, as in the per-window loop.
    power = librosa.feature.melspectrogram(
        y=batch, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    # ref=np.max and the normalisation stay per window.
    return np.stack([to_input(librosa.power_to_db(m, ref=np.max, top_db=TOP_DB)) for m in power])

def file_windows(path):
    return windows_to_inputs(decode(path))

def to_db(u8, store_db):
    # uint8 from make_mels -> dB relative to the clip's loudest point.
    return u8.T.astype(np.float32) / 255.0 * store_db - store_db

class ValWindows:
    # First 5 s of each validation clip. CPU only; each DataLoader worker opens its own memmap.
    def __init__(self, mels_path, starts, frames, store_db, cfg):
        self.mels_path, self.starts, self.frames, self.store_db, self.cfg = mels_path, starts, frames, store_db, cfg
        self.mel = None

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, i):
        if CFG is None:
            configure(self.cfg)
        if self.mel is None:
            self.mel = np.load(self.mels_path, mmap_mode="r")
        s = int(self.starts[i])
        return to_input(to_db(np.array(self.mel[s:s + self.frames]), self.store_db))
"""
(WORK / "adabn_lib.py").write_text(LIB_SRC)
sys.path.insert(0, str(WORK.resolve()))
importlib.invalidate_caches()
import adabn_lib as lib
lib.configure(CFG)


## Soundscape windows
The file list is `noise_index.csv` from make-mels; the audio is read from `train_soundscapes`.


In [ ]:
SETUP_S = time.time() - T_NOTEBOOK
names = pd.read_csv(MELS / "noise_index.csv")["filename"].astype(str).tolist()
files = [DATA / "train_soundscapes" / Path(n).name for n in names]
missing = [str(p) for p in files if not p.exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} of {len(files)} soundscapes missing, e.g. {missing[:3]}")
assert len(set(files)) == len(files), "duplicate soundscape files in noise_index.csv"
print(f"{len(files)} soundscape files, e.g. {names[:2]}")

t0 = time.time()
with mp.get_context("spawn").Pool(NUM_WORKERS, initializer=lib.configure, initargs=(CFG,)) as pool:
    per_file = list(pool.imap(lib.file_windows, files, chunksize=1))
WINDOWS = np.concatenate(per_file)
del per_file
WINDOWS_S = time.time() - t0
n_windows = len(WINDOWS)
print(f"windows {WINDOWS.shape} {WINDOWS.dtype} {WINDOWS.nbytes / 1e9:.2f} GB | {WINDOWS_S:.1f}s")
assert WINDOWS.dtype == np.float32 and WINDOWS.ndim == 3, (WINDOWS.dtype, WINDOWS.shape)
if not SMOKE:
    assert n_windows > MIN_WINDOWS, f"only {n_windows} windows (need > {MIN_WINDOWS})"

order = np.random.RandomState(SEED).permutation(n_windows)     # shuffled across files
n_batches = n_windows // BATCH_SIZE                            # drops the last partial batch
if n_batches == 0:
    raise RuntimeError(f"{n_windows} windows is less than one batch of {BATCH_SIZE}")
print(f"{n_batches} batches of {BATCH_SIZE}, dropping {n_windows - n_batches * BATCH_SIZE} windows")


## Model and validation protocol
This is the same as in our team's baseline v2 training: the split, the primary-label targets, the first 5 s of each clip, `to_db`, `to_input` and `macro_auc`. The cached mels are memory-mapped and read in DataLoader workers, and the model runs on the GPU.


In [ ]:
torch.set_num_threads(TORCH_THREADS)
CKPT = torch.load(MODEL_DIR / "model.pth", map_location="cpu")
model = timm.create_model(CFG["model"], pretrained=False, in_chans=1, num_classes=len(species))
model.load_state_dict(CKPT)
model = model.to(DEVICE).eval()

index = pd.read_csv(MELS / "train_index.csv")
mcfg = json.load(open(MELS / "mels_cfg.json"))
assert "store_db" in mcfg, "old make_mels output (80 dB); rerun make_mels version 2"
for k in ("sr", "n_fft", "hop_length", "n_mels", "fmin", "fmax"):
    assert mcfg[k] == CFG[k], f"mel setting {k} differs from make_mels"
W = mcfg["frames_per_5s"]                                      # 313 frames = 5 s
STORE_DB = float(mcfg["store_db"])                             # train_mels.npy: (total_frames, 128) uint8

sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values
assert CFG["seed"] == SEED, CFG["seed"]
idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
print("train", len(idx_tr), "| val", len(idx_va))

START = index["start"].values

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro")

VAL_BATCH = CFG["batch_size"] * 2
val_set = lib.ValWindows(str(MELS / "train_mels.npy"), START[idx_va], W, STORE_DB, CFG)
y_val = np.zeros((len(idx_va), len(species)), dtype=np.float32)
y_val[np.arange(len(idx_va)), y_idx[idx_va]] = 1.0              # primary label only

@torch.no_grad()
def evaluate(net):
    net.eval()
    loader = torch.utils.data.DataLoader(val_set, batch_size=VAL_BATCH, shuffle=False, drop_last=False,
                                         num_workers=VAL_WORKERS, pin_memory=DEVICE == "cuda")
    preds = []
    for xb in loader:
        logits = net(xb[:, None].to(DEVICE, non_blocking=True))
        preds.append(torch.sigmoid(logits.float()).cpu().numpy())
    preds = np.concatenate(preds)
    assert len(preds) == len(y_val), (len(preds), len(y_val))
    return macro_auc(y_val, preds)


## Validation before adaptation
We evaluate the unadapted model in eval mode, with the same evaluation as after adaptation. The saved checkpoint is the baseline's best epoch, so its AUC should match the baseline's recorded value.


In [ ]:
assert not any(m.training for m in model.eval().modules())
t0 = time.time()
VAL_AUC_BEFORE = evaluate(model)
EVAL_BEFORE_S = time.time() - t0
VAL_AUC_BEFORE_SOURCE = "computed in this run on the unadapted model"
print(f"val AUC before adaptation {VAL_AUC_BEFORE:.4f} | {EVAL_BEFORE_S:.1f}s")

REF_AUC, REF_SOURCE = None, None
if CFG.get("best_val_auc") is not None:
    REF_AUC, REF_SOURCE = float(CFG["best_val_auc"]), "cfg.json best_val_auc"
elif (MODEL_DIR / "history.csv").exists():
    hist = pd.read_csv(MODEL_DIR / "history.csv")
    if "val_auc" in hist.columns and len(hist):
        REF_AUC, REF_SOURCE = float(hist["val_auc"].max()), "history.csv max val_auc"

if REF_AUC is None:
    print("no best val AUC in M1's cfg.json or history.csv: the before value is not checked")
else:
    gap = abs(VAL_AUC_BEFORE - REF_AUC)
    print(f"reference {REF_AUC:.4f} ({REF_SOURCE}) | gap {gap:.4f} (limit {VAL_BEFORE_TOL})")
    if not gap <= VAL_BEFORE_TOL:
        raise RuntimeError(f"val AUC before {VAL_AUC_BEFORE:.4f} differs from {REF_SOURCE} {REF_AUC:.4f} by more than {VAL_BEFORE_TOL}")


## AdaBN
We put the whole model in eval mode, then reset the BatchNorm layers, switch them to a cumulative average (`momentum=None`) and put them in train mode. timm's `BatchNormAct2d` has child modules (`act`, `drop`) that `.train()` would also switch, so we set these back to eval and keep only the BN layers themselves in train mode.


In [ ]:
BN_TYPE = torch.nn.modules.batchnorm._BatchNorm
model.eval()
bn_modules = [(n, m) for n, m in model.named_modules() if isinstance(m, BN_TYPE)]
for _, m in bn_modules:
    m.reset_running_stats()
    m.momentum = None
    m.train()
    for child in m.children():
        child.eval()

assert len(bn_modules) >= 1, "no BatchNorm modules found"
bn_names = {n for n, _ in bn_modules}
active = {n for n, m in model.named_modules() if m.training}
assert active == bn_names, f"modules in train mode differ from the BN set: {sorted(active ^ bn_names)[:5]}"
for n, m in model.named_modules():
    if isinstance(m, torch.nn.modules.dropout._DropoutNd) or type(m).__name__ == "DropPath":
        assert not m.training, f"{n} is in train mode"
print(f"{len(bn_modules)} BatchNorm modules set to adapt | {n_windows} windows | {n_batches} batches")

t0 = time.time()
GUARD_AT = min(GUARD_BATCHES, n_batches)
PROJECTED_MIN = None
with torch.no_grad():
    for b in range(n_batches):
        idx = order[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
        model(torch.from_numpy(WINDOWS[idx])[:, None].to(DEVICE))   # float32, no AMP
        if b + 1 == GUARD_AT:
            gpu_sync()
            per_batch = (time.time() - t0) / GUARD_AT
            remaining = per_batch * (n_batches - GUARD_AT)
            val_estimate = EVAL_BEFORE_S                                # the validation pass already measured here
            elapsed = time.time() - T_NOTEBOOK                          # includes window generation and validation before
            PROJECTED_MIN = (elapsed + remaining + val_estimate) / 60
            print(f"guard: {per_batch:.2f}s per batch | elapsed {elapsed:.0f}s (windows {WINDOWS_S:.0f}s) | "
                  f"adapt left {remaining:.0f}s | val estimate {val_estimate:.0f}s | "
                  f"projected notebook {PROJECTED_MIN:.1f} min (limit {GUARD_MIN:g})", flush=True)
            if PROJECTED_MIN > GUARD_MIN:
                raise RuntimeError(f"projected notebook time {PROJECTED_MIN:.1f} min exceeds {GUARD_MIN:g} min")
        if (b + 1) % 50 == 0:
            print(f"batch {b + 1}/{n_batches} | {time.time() - t0:.0f}s", flush=True)
gpu_sync()
ADAPT_S = time.time() - t0

for n, m in bn_modules:
    assert int(m.num_batches_tracked) == n_batches, f"{n}: {int(m.num_batches_tracked)} != {n_batches}"
print(f"adapted {len(bn_modules)} BN modules on {n_windows} windows ({n_batches} batches) in {ADAPT_S:.1f}s")


## Validation after adaptation, and checks on what changed


In [ ]:
model.eval()
assert not any(m.training for m in model.modules())

t0 = time.time()
VAL_AUC_AFTER = evaluate(model)
EVAL_AFTER_S = time.time() - t0
print(f"val AUC before {VAL_AUC_BEFORE:.4f} ({VAL_AUC_BEFORE_SOURCE}) | after {VAL_AUC_AFTER:.4f} | {EVAL_AFTER_S:.1f}s")

state = {k: v.detach().cpu() for k, v in model.state_dict().items()}   # CKPT is on the CPU
STAT_SUFFIXES = (".running_mean", ".running_var", ".num_batches_tracked")
stat_keys = [k for k in state if k.endswith(STAT_SUFFIXES)]
other_keys = [k for k in state if not k.endswith(STAT_SUFFIXES)]
changed_other = [k for k in other_keys if not torch.equal(state[k], CKPT[k])]
assert not changed_other, f"non-BN-statistic entries changed: {changed_other[:5]}"
mean_keys = [k for k in stat_keys if k.endswith(".running_mean")]
changed_mean = [k for k in mean_keys if not torch.equal(state[k], CKPT[k])]
assert changed_mean, "no BN running_mean changed"
print(f"unchanged non-statistic entries: {len(other_keys)} | running_mean changed in {len(changed_mean)}/{len(mean_keys)} BN modules")


## Save


In [ ]:
t0 = time.time()
model.eval()
assert all(v.device.type == "cpu" for v in state.values()), "model.pth must hold CPU tensors"
torch.save(state, WORK / "model.pth")
for name in ("cfg.json", "species.json"):
    shutil.copyfile(MODEL_DIR / name, WORK / name)

fresh = timm.create_model(CFG["model"], pretrained=False, in_chans=1, num_classes=len(species))
loaded = torch.load(WORK / "model.pth", map_location="cpu")
fresh.load_state_dict(loaded, strict=True)
final = state
assert all(torch.equal(fresh.state_dict()[k], final[k]) for k in final), "reloaded model differs"
print("model.pth reloaded with strict=True")
SAVE_S = time.time() - t0
TOTAL_S = time.time() - T_NOTEBOOK

results = dict(val_auc_before=float(VAL_AUC_BEFORE), val_auc_before_source=VAL_AUC_BEFORE_SOURCE,
               val_auc_after=float(VAL_AUC_AFTER), val_auc_before_reference=REF_AUC,
               val_auc_before_reference_source=REF_SOURCE, n_files=len(files),
               n_windows=int(n_windows), n_batches=int(n_batches), n_bn_modules=len(bn_modules),
               setup_seconds=SETUP_S, windows_seconds=WINDOWS_S, val_before_seconds=EVAL_BEFORE_S,
               adapt_seconds=ADAPT_S, val_after_seconds=EVAL_AFTER_S, save_seconds=SAVE_S,
               total_seconds=TOTAL_S, device=DEVICE, device_name=DEVICE_NAME)
with open(WORK / "adabn_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(results)
print(f"TIMING | setup {SETUP_S:.1f}s | windows {WINDOWS_S:.1f}s | val before {EVAL_BEFORE_S:.1f}s | adapt {ADAPT_S:.1f}s | "
      f"val after {EVAL_AFTER_S:.1f}s | save {SAVE_S:.1f}s | total {TOTAL_S:.1f}s ({TOTAL_S / 60:.1f} min)")
